In [0]:
# ================================================
# CELL 1 - READ FROM AGGREGATION OUTPUT
# ================================================
from pyspark.sql.functions import col, sum as spark_sum

FORCE_NAME = "nottinghamshire"
OUTPUT_PATH = f"/Volumes/workspace/default/crime_project/outputs/{FORCE_NAME}_reporting"

df_final = spark.table(
    "default.nottinghamshire_reporting"
)

print(f"✅ Rows loaded: {df_final.count():,}")
print(f"✅ Columns: {df_final.columns}")

In [0]:
# ================================================
# CELL 2 - FINAL VALIDATION
# ================================================

print("=== NULL CHECK ===")
for field in df_final.columns:
    null_count = df_final.filter(
        col(field).isNull()
    ).count()
    status = "✅" if null_count == 0 else "⚠️"
    print(f"{status} {field}: {null_count} nulls")

print("\n=== GRAIN DUPLICATE CHECK ===")
grain_dupes = df_final.groupBy(
    "lsoa_code", "year",
    "month_num", "crime_type"
).count() \
.filter(col("count") > 1)
print(f"Duplicate rows at grain: {grain_dupes.count()}")

print("\n=== TOTAL CRIMES CHECK ===")
total = df_final.agg(
    spark_sum("total_crimes")
).collect()[0][0]
print(f"Total crimes: {total:,}")

print("\n=== YEAR CHECK ===")
df_final.groupBy("year") \
    .agg(spark_sum("total_crimes").alias("total_crimes")) \
    .orderBy("year") \
    .show()

print("\n=== MONTHLY TOTALS ===")
df_final.groupBy("year", "month_num", "month_name") \
    .agg(spark_sum("total_crimes").alias("total_crimes")) \
    .orderBy("year", "month_num") \
    .show(100)

In [0]:
# ================================================
# CELL 3 - EXPORT AS CSV
# ================================================

df_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(OUTPUT_PATH)

print(f"✅ EXPORT COMPLETE")
print(f"✅ Location: {OUTPUT_PATH}")
print(f"✅ Total rows exported: {df_final.count():,}")

In [0]:
# ================================================
# CELL 4 - VERIFY EXPORT
# ================================================

df_verify = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(OUTPUT_PATH)

print("=== EXPORT VERIFICATION ===")
print(f"✅ Rows in exported file: {df_verify.count():,}")
print(f"✅ Columns: {df_verify.columns}")

total_verify = df_verify.agg(
    spark_sum("total_crimes")
).collect()[0][0]
print(f"✅ Total crimes: {total_verify:,}")

print("\n=== YEAR CHECK ===")
df_verify.groupBy("year") \
    .agg(spark_sum("total_crimes").alias("total_crimes")) \
    .orderBy("year") \
    .show()

display(df_verify.limit(10))

In [0]:
# ================================================
# CELL 5 - RENAME FILE
# ================================================

files = dbutils.fs.ls(OUTPUT_PATH)
part_file = [f.path for f in files 
             if f.name.startswith("part-")][0]

print(f"Found file: {part_file}")

dbutils.fs.cp(
    part_file,
    "/Volumes/workspace/default/crime_project/outputs/nottinghamshire_crime_2020_2021.csv"
)

print("✅ File saved as: nottinghamshire_crime_2020_2021.csv")
print("✅ Location: /Volumes/workspace/default/crime_project/outputs/")